In [1]:
import os
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
print("Starting script...")

TF_AVAILABLE = False
print("TensorFlow disabled; using fallback predictions.")

PLATFORM = "kaggle"  # local/kaggle
NEEDTRAIN = False
DATATYPE = ["spe", "img"]  # unused in this lightweight version
STAGETRAIN = [2, 3]
STAGETEST = 3
LOAD_MODELS_FROM = "models2024032602"

if PLATFORM == "local":
    LOAD_MODELS_FROM = f"./input/{LOAD_MODELS_FROM}"
elif PLATFORM == "kaggle":
    LOAD_MODELS_FROM = f"/kaggle/input/{LOAD_MODELS_FROM}"

SEED = 2024
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

if PLATFORM == "local":
    df = pd.read_csv("./input/hms-harmful-brain-activity-classification/train.csv")
else:  # kaggle
    df = pd.read_csv(
        "/kaggle/input/hms-harmful-brain-activity-classification/train.csv"
    )
TARGETS = df.columns[-6:]  # vote columns
print("Train shape:", df.shape)
print("Targets", list(TARGETS))

# Increase Laplace smoothing to reduce over‑confident priors
smooth = 5
global_counts = df[TARGETS].sum() + smooth * len(TARGETS)
global_probs = global_counts / global_counts.sum()
print("Global class prior probabilities (smoothed):")
for col, prob in zip(TARGETS, global_probs):
    print(f"  {col}: {prob:.5f}")

patient_counts = df.groupby("patient_id")[TARGETS].sum() + smooth
patient_probs = patient_counts.div(patient_counts.sum(axis=1), axis=0)
patient_total_votes = patient_counts.sum(axis=1)  # smoothed votes per patient

eeg_counts = df.groupby("eeg_id")[TARGETS].sum() + smooth
eeg_probs = eeg_counts.div(eeg_counts.sum(axis=1), axis=0)
eeg_total_votes = eeg_counts.sum(axis=1)  # smoothed votes per EEG

if not NEEDTRAIN:
    if PLATFORM == "local":
        test = pd.read_csv("./input/hms-harmful-brain-activity-classification/test.csv")
    else:  # kaggle
        test = pd.read_csv(
            "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"
        )
    print("Test shape", test.shape)

    # Raise thresholds so we only use specific priors when we have enough evidence
    VOTE_THRESHOLD_EEG = 20  # use EEG prior when at least 20 votes
    VOTE_THRESHOLD_PATIENT = 10  # use patient prior when at least 10 votes

    # Shift blending weights toward the safe global prior
    WEIGHT_EEG = 0.2
    WEIGHT_PATIENT = 0.3
    WEIGHT_GLOBAL = 1.0 - (WEIGHT_EEG + WEIGHT_PATIENT)  # 0.5

    pred_list = []
    for _, row in test.iterrows():
        eid = row["eeg_id"]
        pid = row["patient_id"]

        has_eeg = (eid in eeg_probs.index) and (
            eeg_total_votes.loc[eid] >= VOTE_THRESHOLD_EEG
        )
        has_patient = (pid in patient_probs.index) and (
            patient_total_votes.loc[pid] >= VOTE_THRESHOLD_PATIENT
        )

        probs = np.zeros(len(TARGETS), dtype=float)

        if has_eeg:
            probs += WEIGHT_EEG * eeg_probs.loc[eid].values
        if has_patient:
            probs += WEIGHT_PATIENT * patient_probs.loc[pid].values

        probs += WEIGHT_GLOBAL * global_probs.values

        pred_list.append(probs)

    pred_array = np.vstack(pred_list).astype(np.float32)

    eps = 1e-12
    pred_array = pred_array + eps
    pred_array = pred_array / pred_array.sum(axis=1, keepdims=True)

    sub = pd.DataFrame({"eeg_id": test["eeg_id"].values})
    sub[TARGETS] = pred_array
    sub_path = "submission.csv"
    sub.to_csv(sub_path, index=False)
    print(f"Submission written to {sub_path}")
    print("Submission shape", sub.shape)
    print(sub.head())

Starting script...
TensorFlow disabled; using fallback predictions.
Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
Global class prior probabilities (smoothed):
  seizure_vote: 0.12056
  lpd_vote: 0.15755
  gpd_vote: 0.17107
  lrda_vote: 0.13312
  grda_vote: 0.14811
  other_vote: 0.26958
Test shape (9850, 3)
Submission written to submission.csv
Submission shape (9850, 7)
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  2578018731      0.076699  0.105207  0.316949   0.086197   0.161881   
1  2578018731      0.076699  0.105207  0.316949   0.086197   0.161881   
2  2578018731      0.076699  0.105207  0.316949   0.086197   0.161881   
3  2578018731      0.076699  0.105207  0.316949   0.086197   0.161881   
4  1706196508      0.118830  0.209883  0.110542   0.135437   0.101325   

   other_vote  
0    0.253067  
1    0.253067  
2    0.253067  
3    0.253067  
4    0.323981  
